In [1]:
source("helpers/init.r")

# 0 - Define cohorts
#
# This notebook is ONE of TWO ways to configure the pipeline:
#
#   Option A (this notebook): Edit the parameters below and run 0_cohort.ipynb.
#                            It generates cohorts.txt automatically.
#
#   Option B (recommended):  Prepare a cohorts CSV directly and run:
#                            bash run.sh your_cohorts.csv
#                            Columns: sampleId, cohort, ref, primaryTumorLocation, title


In [2]:
clinical <- fread(paste0(BASE_DATA, "clinical.txt"))
markers <- fread(paste0(BASE_DATA, "markers.txt"))

In [3]:
tmp <- clinical %>% inner_join(markers %>% select(sampleId), by = "sampleId")

In [4]:
test_name <- "Prostate"
test_ids  <- tmp %>% filter(primaryTumorLocation == "Prostate") %>% pull(sampleId)  # add sample IDs here

In [5]:
ref_name <- "Lung"
ref_ids  <- tmp %>% filter(primaryTumorLocation == "Lung") %>% pull(sampleId)   # add sample IDs here

In [6]:
location <- "Prostate"

if (!location %in% names(schema_map)) {
  stop(paste0("'", location, "' is not a valid primaryTumorLocation.\nValid options:\n  ",
              paste(sort(names(schema_map)), collapse="\n  ")))
}

# 1 - Save

In [8]:
doids_df <- clinical %>% select(sampleId, doids)

cohorts <- bind_rows(
  data.frame(sampleId = test_ids, cohort = test_name, ref = FALSE),
  data.frame(sampleId = ref_ids,  cohort = ref_name,  ref = TRUE)
) %>%
  mutate(primaryTumorLocation = location,
         title = "Prostate Cancer") %>%
  left_join(doids_df, by = "sampleId")
fwrite(cohorts, paste0(READY_DATA, "cohorts.txt"))